# Runtime y scheduler: control de ejecución · 2.1.4

Input value=3 → scheduler acota el actor → System conserva resultado y evidencia. Runtime define tiempo/reintentos/turnos/Tools; plazo raíz acota la ejecución compuesta. Un timeout no garantiza detener un trabajador. Las ramas del grafo son una decisión distinta de max_concurrency del actor.

In [2]:
import pydantic

In [3]:
import agentic_systems as toolkit

In [4]:
class State(pydantic.BaseModel):
    value: int

In [5]:
@toolkit.tool
def echo(value: int) -> dict:
    return {"value": value}

In [6]:
def run():
    execution = toolkit.runtime(
        provider="python-runtime",
        timeout_s=30,
        max_turns=2,
        max_tool_calls=1,
        max_retries=0,
        backoff_s=0,
        max_concurrency=1,
    )
    adapter = toolkit.framework("native")
    actor = toolkit.Agent(
        "bounded_operator",
        runtime=execution,
        framework=adapter,
        skills=(toolkit.skill(name="echo", tools=(echo,)),),
        policy={"repair": False, "trace": "compact"},
        contract={
            "tool_expectation": {"all_of": ["echo"]},
            "output_schema": State.model_json_schema(),
        },
    )
    graph = toolkit.Graph(State)
    graph.add_node("echo", actor)
    graph.add_edge(toolkit.START, "echo")
    graph.add_edge("echo", toolkit.END)
    system = toolkit.System("bounded_system", graph=graph)
    deadline_s = 60
    result = system.run(State(value=3), timeout_s=deadline_s)
    assert result.ok and result.data == {"value": 3}, result.to_dict()
    negative = system.run({"value": "invalid"}, timeout_s=deadline_s)
    assert not negative.ok
    assert isinstance(execution, pydantic.BaseModel)
    assert isinstance(adapter, pydantic.BaseModel)
    for bad in (
        {"max_retries": -1},
        {"max_concurrency": 0},
        {"timeout_s": float("inf")},
    ):
        try:
            toolkit.runtime(provider="python-runtime", **bad)
        except pydantic.ValidationError:
            pass
        else:
            raise AssertionError("Invalid controls must fail before effects")
    return {
        "input": {"value": 3},
        "process": "runtime + scheduler caps -> actor -> System limits",
        "output": result.to_dict(),
        "negative": negative.to_dict(),
        "runtime": execution.inspect(),
        "timeout_s": deadline_s,
        "inference_requests": 0,
    }

In [7]:
report = run()
result = toolkit.RunResult(**report["output"])
print(result.human_results())
print(report["runtime"])

bounded_system: COMPLETED
Execution: run-90b5d6902ba149b6b2345b6794af5853
Input: {"value":3}
Output: {"value":3}
Process:
- Agent bounded_operator: completed
- Tool echo: completed -> {"value":3}
{'provider': 'python-runtime', 'model': 'python-runtime', 'region': None, 'endpoint': None, 'api_key_configured': False, 'session_configured': False, 'authentication_mode': 'provider-configuration', 'timeout_s': 30, 'max_retries': 0, 'max_tool_calls': 1, 'max_turns': 2, 'max_tokens': None, 'backoff_s': 0, 'max_concurrency': 1}
